In [18]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from skorch import NeuralNetBinaryClassifier
from skorch.callbacks import EarlyStopping

import copy
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, 
    average_precision_score,
    roc_auc_score, 
    precision_score, 
    recall_score, 
    accuracy_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    log_loss
)

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(f"current working directory: {os.getcwd()}")

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"device: {device}")

current working directory: /Users/cbrown/code/cse_284
device: mps


# Import and Preprocess

In [19]:
# paths
geno_path = '/Users/cbrown/code/cse_284/data/processed/Effect allele count.csv'
pheno_path = "data/raw/GenRisk_ImputedData_CRC_Samples.txt"
drop_ids_path = '/Users/cbrown/code/cse_284/data/processed/GenRisk_all_CRC_SNPs_perm.clean.vcf.dropped.txt'

# import genotypes
df_geno = pd.read_csv(geno_path)
df_geno = df_geno.set_index('IID')
drop_ids = pd.read_csv(drop_ids_path, header=None).to_numpy().ravel()
df_geno = df_geno.loc[~df_geno.index.isin(drop_ids)]        # filter out samples dropped due to genotype issues in preprocessing

# import phenotypes
df_pheno = pd.read_csv(pheno_path, sep='\t')
df_pheno = df_pheno.set_index('ID')
df_pheno = df_pheno.reindex(df_geno.index)

print(df_geno.shape)
print(df_pheno.shape)

(4104, 204)
(4104, 1)


# Helper Functions

In [20]:
def evaluate_model(model_name, y_true, y_pred, y_prob):
    """
    Calculates key classification metrics and returns them as a dictionary.
    """

    pr_auc = average_precision_score(y_true, y_prob, pos_label='Colorectal')
    roc_auc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred, pos_label='Colorectal')
    precision = precision_score(y_true, y_pred, pos_label='Colorectal')
    recall = recall_score(y_true, y_pred, pos_label='Colorectal')
    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    ll = log_loss(y_true, y_prob)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=['Control', 'Colorectal']).ravel()
    specificity = tn / (tn + fp)
    
    metrics_dict = {
        'Model': model_name,
        'PR_AUC': round(pr_auc, 4),
        'F1_Score': round(f1, 4),
        'ROC_AUC': round(roc_auc, 4),
        'Precision': round(precision, 4),
        'Recall_Sensitivity': round(recall, 4),
        'Specificity': round(specificity, 4),
        'Accuracy': round(accuracy, 4),
        'Balanced Accuracy': round(balanced_accuracy, 4),
        'Log Loss': round(ll, 4)
    }
    
    return metrics_dict

In [21]:
def random_parameter_search(model, param_distributions, cv_strategy, X_train_val, y_train_val, n_iter=30):
    """
    Run a random search over paremter grid
    """
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring='balanced_accuracy',
        cv=cv_strategy,        # 5-fold stratified CV
        n_jobs=-1,
        verbose=1,
        random_state=42
    )

    # fit model
    random_search.fit(X_train_val, y_train_val)
    print(f"Best Parameters Found: {random_search.best_params_}")
    print(f"Best Cross-Validation Balanced Accuracy: {random_search.best_score_:.4f}")

    return random_search

def best_model_predict(random_search, X_test):
    """
    Return predictions and prediction probabilities on test data using best model from random search
    """
    best_lr = random_search.best_estimator_
    y_test_probs = best_lr.predict_proba(X_test)[:, 1]
    y_test_preds = best_lr.predict(X_test)
    return y_test_preds, y_test_probs

def get_classification_metrics(model_name, random_search, X_test, y_test):
    """
    Return results on hold-out test data using best model from random search
    """
    # get predictions and prediction probs from best model
    y_test_preds, y_test_probs = best_model_predict(random_search, X_test)

    print("Classification Report:")
    print(classification_report(y_test, y_test_preds, target_names=['Control', 'Case']))

    classification_metrics = evaluate_model(
        model_name, 
        y_test, y_test_preds, 
        y_test_probs
    )
    return classification_metrics

# Generate train test split

In [22]:
# Generate X genotype matrix and y phenotype array
X = df_geno.iloc[:,5:].copy()
y = np.array(df_pheno.values).ravel()

In [23]:
# stratified train test split for 20% hold-out set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y, 
    random_state=42
)
print(f"Outer split shape: {X_train_val.shape}")
print(f"Hold-out test set: {X_test.shape}")

Outer split shape: (3283, 199)
Hold-out test set: (821, 199)


In [24]:
model_results = []

# PRS model

In [25]:
PRS_df = pd.read_csv("results/PRS/4_PRS_calculation/score_final_w_labels.txt", sep="\t", index_col=0)
X_train_val_PRS = PRS_df.loc[X_train_val.index, 'SCORE1_AVG_scale'].to_numpy().reshape(-1,1)
X_test_PRS = PRS_df.loc[X_test.index, 'SCORE1_AVG_scale'].to_numpy().reshape(-1,1)

In [26]:
PRS_model = LogisticRegression(class_weight='balanced', C=np.inf, random_state=42)

PRS_model.fit(X_train_val_PRS, y_train_val)

y_test_probs = PRS_model.predict_proba(X_test_PRS)[:, 1]
y_test_preds = PRS_model.predict(X_test_PRS)

print(classification_report(y_test, y_test_preds, target_names=['Control', 'Case']))

classification_metrics = evaluate_model('PRS', y_test, y_test_preds, y_test_probs)
model_results.append(classification_metrics)

              precision    recall  f1-score   support

     Control       0.39      0.58      0.46       248
        Case       0.77      0.60      0.67       573

    accuracy                           0.59       821
   macro avg       0.58      0.59      0.57       821
weighted avg       0.65      0.59      0.61       821



# Logistic Regression Model

In [27]:
lr_model = LogisticRegression(penalty='elasticnet', class_weight='balanced', random_state=42, solver='saga', max_iter=500)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_distributions = {
    'l1_ratio': [0, 0.2, 0.4, 0.6, 0.8, 1],
    'C': [0.1, 0.5, 1, 2, 4]
}

# run random parameter search to find best model
random_search = random_parameter_search(lr_model, param_distributions, cv_strategy, X_train_val, y_train_val)

# log results using best model on test set
classification_metrics = get_classification_metrics('Logistic Regression', random_search, X_test, y_test)
model_results.append(classification_metrics)

top_5_snps = pd.Series(random_search.best_estimator_.coef_.ravel(), index=X.columns).nlargest(5)
print("\n--- Top 5 Most Important SNPs ---")
print(top_5_snps)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


/Users/cbrown/miniconda3/envs/ml_env/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/cbrown/miniconda3/envs/ml_env/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/cbrown/miniconda3/envs/ml_env/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/cbrown/miniconda3/envs/ml_env/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/cbrown/miniconda3/envs/ml_env/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.w

Best Parameters Found: {'l1_ratio': 0.2, 'C': 1}
Best Cross-Validation Balanced Accuracy: 0.5596
Classification Report:
              precision    recall  f1-score   support

     Control       0.36      0.51      0.42       248
        Case       0.74      0.60      0.66       573

    accuracy                           0.57       821
   macro avg       0.55      0.56      0.54       821
weighted avg       0.62      0.57      0.59       821


--- Top 5 Most Important SNPs ---
6:106482613_T_C_T    0.505571
8:117630683_A_C_A    0.387626
12:4388271_C_T_C     0.283775
8:117790914_C_A_C    0.260926
9:34107505_C_T_C     0.228766
dtype: float64


# Random Forest Model

In [28]:
# Model
rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_distributions = {
    'n_estimators': [300, 500, 800, 1000],
    'max_features': ['sqrt', 'log2', 20, 50],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10]
}

# run random parameter search to find best model
random_search = random_parameter_search(rf_model, param_distributions, cv_strategy, X_train_val, y_train_val)

# log results using best model on test set
classification_metrics = get_classification_metrics('Random Forest', random_search, X_test, y_test)
model_results.append(classification_metrics)

feature_importances = pd.Series(random_search.best_estimator_.feature_importances_, index=X.columns)
top_5_snps = feature_importances.nlargest(5)
print("\n--- Top 5 Most Important SNPs ---")
print(top_5_snps)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters Found: {'n_estimators': 1000, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': 50, 'max_depth': 5}
Best Cross-Validation Balanced Accuracy: 0.5302
Classification Report:
              precision    recall  f1-score   support

     Control       0.37      0.22      0.27       248
        Case       0.71      0.84      0.77       573

    accuracy                           0.65       821
   macro avg       0.54      0.53      0.52       821
weighted avg       0.61      0.65      0.62       821


--- Top 5 Most Important SNPs ---
10:80819132_A_G_A    0.023993
20:47340117_A_G_A    0.015652
20:6699595_T_G_T     0.015485
8:128414892_T_C_T    0.014765
1:22503282_G_C_G     0.014662
dtype: float64


# NN Model

In [29]:
X_train_val_np = X_train_val.values.astype(np.float32)
y_train_val_np = np.asarray([1 if x == 'Colorectal' else 0 for x in y_train_val], dtype=np.float32)

X_test_np = X_test.values.astype(np.float32)

# balance class weights
num_negatives = (y_train_val_np == 0).sum()
num_positives = (y_train_val_np == 1).sum()
pos_weight = torch.tensor([num_negatives / num_positives], dtype=torch.float32)

In [30]:
# model
class PRSPredictor(nn.Module):
    def __init__(self, input_dim=199, hidden_1=64, hidden_2=32, dropout_rate=0.4):
        super(PRSPredictor, self).__init__()
        
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, hidden_1),
            nn.BatchNorm1d(hidden_1),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        
        self.layer2 = nn.Sequential(
            nn.Linear(hidden_1, hidden_2),
            nn.BatchNorm1d(hidden_2),
            nn.ReLU(),
            nn.Dropout(max(0, dropout_rate - 0.1)) # Keep second dropout slightly lower
        )
        
        self.output = nn.Linear(hidden_2, 1)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.output(x)
        return x

In [31]:
# early stopping
early_stopping = EarlyStopping(
    monitor='valid_loss', 
    patience=15, 
    lower_is_better=True
)

# wrap model in skorch
net = NeuralNetBinaryClassifier(
    module=PRSPredictor,
    module__input_dim=199,
    criterion=nn.BCEWithLogitsLoss,
    criterion__pos_weight=pos_weight,
    optimizer=torch.optim.Adam,
    max_epochs=100,
    callbacks=[early_stopping],
    verbose=0,
    device=device
)

param_distributions = {
    'module__hidden_1': [32, 64, 128],
    'module__hidden_2': [16, 32, 64],
    'module__dropout_rate': [0.3, 0.4, 0.5],
    'lr': [0.0005, 0.001, 0.005],
    'batch_size': [16, 32, 64]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Starting randomize search for NN with 5-fold CV")
random_search = random_parameter_search(net, param_distributions, cv_strategy, X_train_val_np, y_train_val_np)

# make predictions on hold-out data using best model
best_nn = random_search.best_estimator_
y_test_probs = best_nn.predict_proba(X_test_np)[:, 1]
y_test_preds = best_nn.predict(X_test_np)
y_test_preds = ['Colorectal' if x == 1 else 'Control' for x in y_test_preds]

classification_metrics = evaluate_model('NN', y_test, y_test_preds, y_test_probs)
model_results.append(classification_metrics)

Starting randomize search for NN with 5-fold CV
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters Found: {'module__hidden_2': 16, 'module__hidden_1': 128, 'module__dropout_rate': 0.3, 'lr': 0.001, 'batch_size': 16}
Best Cross-Validation Balanced Accuracy: 0.5447


In [32]:
results_df = pd.DataFrame(model_results)
results_df = results_df.sort_values(by='PR_AUC', ascending=False)
print(results_df.to_string(index=False))

results_df.to_csv('results/tables/prs_model_comparison_metrics.csv', index=False)

              Model  PR_AUC  F1_Score  ROC_AUC  Precision  Recall_Sensitivity  Specificity  Accuracy  Balanced Accuracy  Log Loss
                 NN  0.3937    0.3517   0.4093     0.3705              0.3347       0.7539    0.6273             0.5443    1.9680
      Random Forest  0.2605    0.2734   0.5649     0.3673              0.2177       0.8377    0.6504             0.5277    0.6741
Logistic Regression  0.2499    0.4186   0.5832     0.3559              0.5081       0.6021    0.5737             0.5551    0.7002
                PRS  0.2408    0.4630   0.6171     0.3850              0.5806       0.5986    0.5932             0.5896    0.6710
